# Tính năng 4 – Trang **Đánh giá mô hình** (phần 1: huấn luyện và so sánh)

Mô hình nào được chọn và vì sao, sai số trên tập test, thực tế so với dự đoán, đặc trưng quan trọng, giá trị gia tăng của kỹ năng, sai số theo vai trò. Phần kiểm định độ tin cậy và chuẩn production của cùng trang ở notebook tính năng 5.

Mã nguồn: `src/career_market/salary_model.py`, `pages/evaluation.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from career_market.data.dataset import load_dataset
from career_market.model.estimators import MODEL_LABELS, build_model, feature_frame
from career_market.model.training import load_report, load_skill_value, scores
from career_market.paths import artifacts_dir

frame = load_dataset()
report = load_report()
parts = {s: frame[frame["split"] == s] for s in ["train", "valid", "test"]}
print({s: len(p) for s, p in parts.items()})

## 1. Đầu vào – và những gì cố ý **không** đưa vào
Đầu vào: vai trò, khu vực, cấp bậc, **nền tảng đăng tin, hình thức làm việc, loại công ty**, số năm kinh nghiệm, năm dữ liệu, 4 cờ từ khóa chức danh, 69 cột kỹ năng. Không đưa tên công ty, tiêu đề gốc, chuỗi lương: nếu có tên công ty, mô hình sẽ học "công ty X trả cao" – vô dụng với công ty mới.

In [ ]:
X = feature_frame(parts["train"])
print("Số cột đầu vào trước khi mã hóa:", X.shape[1])
X.iloc[:3, :12]

## 2. Bốn ứng viên – như bốn cách con người định lương

| Mô hình | Giống như ngoài đời | Điểm yếu |
|---|---|---|
| **Mốc trung vị** | Ai hỏi cũng trả lời "khoảng 25 triệu" | Thước đo tối thiểu |
| **Ridge** | Bảng giá cộng dồn: Backend +2, mỗi năm +3, AWS +1 | Không hiểu "tùy hoàn cảnh" |
| **Random Forest** | Hỏi 300 chuyên viên, mỗi người xem một phần hồ sơ, rồi lấy trung bình | Ngoài đời không ai có 300 góc nhìn độc lập; từng người dễ "thuộc lòng" |
| **Gradient Boosting** | Hội đồng nối tiếp: người sau chỉ sửa chỗ người trước sai | Khó giải thích hơn Ridge |

**Ba nguyên tắc huấn luyện:** học trên logarit của lương (lệch 5 triệu với người lương 10 triệu nghiêm trọng hơn với người 100 triệu); chọn mô hình bằng MAE trên valid, khóa tiêu chí trước; tập test chỉ mở một lần.

In [ ]:
y = parts["train"]["salary"]
print(f"Skew của lương: {y.skew():.2f} → sau logarit: {np.log(y).skew():.2f}")
rows = [{"Mô hình": MODEL_LABELS[n], "Siêu tham số": t["params"], "MAE valid": t["valid_MAE"]} for n, trials in report["tuning"].items() for t in trials]
tuning = pd.DataFrame(rows)
tuning.assign(**{"Chọn": np.where(tuning.groupby("Mô hình")["MAE valid"].transform("min") == tuning["MAE valid"], "✓", "")})

## 3. Luật đời thường: thêm kinh nghiệm hoặc kỹ năng không làm giảm lương

**Vấn đề (06/11):** thêm Java vào hồ sơ Backend 5 năm làm lương dự đoán giảm từ 47,7 xuống 42,3 triệu. Mô hình không lỗi lập trình mà học đúng dữ liệu: cột kỹ năng là **yêu cầu của tin**, và tin liệt kê nhiều kỹ năng thường là tin junior/tuyển hàng loạt. Bảng dưới kiểm tra điều đó trên dữ liệu thật hiện tại:

In [ ]:
backend = frame[frame["role"] == "Backend Developer"]
bins = pd.cut(backend["skill_count"], [-1, 0, 2, 4, 6, 30], labels=["0", "1–2", "3–4", "5–6", "7+"])
backend.groupby(bins, observed=False)["salary"].agg(["size", "median"]).rename(columns={"size": "Số tin", "median": "Trung vị lương"}).T

**Lựa chọn:** bỏ đặc trưng "số kỹ năng"; Gradient Boosting có ràng buộc đơn điệu tăng cho kinh nghiệm và 69 cột kỹ năng; Ridge chặn hệ số kỹ năng ≥ 0; Random Forest được bọc "bao đơn điệu". Kiểm tra trên mô hình đã lưu:

In [ ]:
from career_market.model.advisor import predict_profile

base = {"role": "Backend Developer", "region": "Hà Nội", "experience_years": 5}
pd.DataFrame([{"Mô hình": MODEL_LABELS[n],
               "Không kỹ năng": predict_profile({**base, "skills": []}, n)["predicted"],
               "+ Java": predict_profile({**base, "skills": ["Java"]}, n)["predicted"],
               "+ Java, Spring Boot, Docker, AWS": predict_profile({**base, "skills": ["Java", "Spring Boot", "Docker", "AWS"]}, n)["predicted"]}
              for n in ["GradientBoosting", "RandomForest", "Ridge"]]).set_index("Mô hình")

## 4. Tái lập kết quả: huấn luyện lại Gradient Boosting với tham số đã chọn

In [ ]:
params = report["models"]["GradientBoosting"]["params"]
model = build_model("GradientBoosting", **params).fit(feature_frame(parts["train"]), parts["train"]["salary"])
again = {s: scores(parts[s]["salary"], model.predict(feature_frame(parts[s])))["MAE"] for s in ["train", "valid", "test"]}
pd.DataFrame({"Fit lại trong notebook": again, "Trong metrics.json": {s: report["models"]["GradientBoosting"][s]["MAE"] for s in again}})

## 5. Kết quả trên ba tập

In [ ]:
rows = [{"Mô hình": v["label"], "MAE train": v["train"]["MAE"], "MAE valid": v["valid"]["MAE"], "MAE test": v["test"]["MAE"],
         "RMSE test": v["test"]["RMSE"], "R² test": v["test"]["R2"], "MAPE test %": v["test"]["MAPE"]} for v in report["models"].values()]
display(pd.DataFrame(rows).set_index("Mô hình"))
print("Được chọn theo MAE valid:", MODEL_LABELS[report["selected"]])
charts.model_comparison(report);

- Gradient Boosting thấp nhất trên valid → được chọn **trước khi mở test**; trên test vẫn tốt nhất.
- Khoảng cách train – valid của Gradient Boosting lớn nhất: với 1.169 tin train, mô hình học thuộc một phần.
- Các con số cao hơn phiên bản có lẫn tin sinh tự động (MAE 5,82) vì tin thật khó đoán hơn – hai con số không so trực tiếp được.

## 6. Mô hình sai ở đâu?

In [ ]:
pred = pd.read_csv(artifacts_dir() / "test_predictions.csv")
error = (pred["predicted"] - pred["salary"]).abs() / pred["salary"]
print(f"Một nửa số tin test lệch dưới {error.median():.1%}; {(error <= 0.2).mean():.0%} lệch không quá 20%")
charts.actual_vs_predicted(pred);

In [ ]:
charts.error_by_role(pred);

Sai số lớn nhất ở Tech Lead/Architect và quản lý: lương ở startup 20 người khác xa tập đoàn 5.000 người dù cùng chức danh – quy mô công ty không có trong dữ liệu.

## 7. Yếu tố quyết định lương và kỹ năng đáng học
**Permutation importance:** xáo trộn một cột rồi xem mô hình kém đi bao nhiêu – như cho một cầu thủ nghỉ một trận để biết anh ta quan trọng thế nào.

In [ ]:
charts.feature_importance(report);

In [ ]:
value = load_skill_value()
display(value[["skill", "postings", "premium_median", "model_uplift"]].rename(columns={
    "skill": "Kỹ năng", "postings": "Số tin", "premium_median": "Chênh trung vị thô", "model_uplift": "Mức tăng mô hình"}).head(12))
print("Kỹ năng có mức tăng âm:", int((value["model_uplift"] < 0).sum()))
charts.skill_uplift(value);

⚠️ Mức tăng là phép thử "nếu như" trên mô hình học từ tin tuyển dụng – **tương quan, không phải nhân quả**.